# 3.45 — Feature Engineering

Feature engineering changes the representation of raw inputs so a simple model can see the signal it would otherwise miss. In this lesson, we build feature maps `z = φ(x)` from scratch — normalized scales, interactions, bins, counts, and validation-aware complexity checks — using only NumPy and Matplotlib.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build feature engineering one idea at a time. Run each cell in order and inspect the intermediate values: the goal is not to memorize tricks, but to see why changing coordinates can make empirical risk easier to reduce and easier to validate. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized arithmetic, and small linear models.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic examples.

### 1. Feature maps turn raw inputs into model-visible coordinates

A raw input `x` is not sacred. Feature engineering chooses a map `z = φ(x)` before the model sees the example. If the target depends on a curved pattern but the model is linear in its inputs, the right feature map can make the relationship linear in the transformed coordinates.

In [ ]:
x_w = np.array([-2., -1., 0., 1., 2.])  # one raw scalar input.
y_w = 1.0 + 2.0 * x_w + 0.5 * x_w**2  # target with a linear and quadratic part.
Z_raw_w = np.c_[np.ones_like(x_w), x_w]  # intercept + raw x only.
Z_quad_w = np.c_[np.ones_like(x_w), x_w, x_w**2]  # intercept + x + engineered x^2.
print("raw design shape:", Z_raw_w.shape)
print("quadratic design shape:", Z_quad_w.shape)
print("targets:", y_w)

▶ What you'll see: the engineered matrix has one extra column, `x²`, which is the missing coordinate behind the curved target.

In [ ]:
coef_raw_w = np.linalg.lstsq(Z_raw_w, y_w, rcond=None)[0]  # best linear fit without x^2.
coef_quad_w = np.linalg.lstsq(Z_quad_w, y_w, rcond=None)[0]  # best linear fit with x^2.
pred_raw_w = Z_raw_w @ coef_raw_w
pred_quad_w = Z_quad_w @ coef_quad_w
mse_raw_w = float(np.mean((y_w - pred_raw_w) ** 2))
mse_quad_w = float(np.mean((y_w - pred_quad_w) ** 2))
print("raw MSE:", round(mse_raw_w, 3), "quadratic-feature MSE:", round(mse_quad_w, 3))
assert round(mse_raw_w, 3) == 0.7
assert round(mse_quad_w, 12) == 0.0

▶ What you'll see: adding `x²` reduces the training error from 0.700 to exactly 0 on this toy curve.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.scatter(x_w, y_w, color="black", label="data")
plt.plot(x_w, pred_raw_w, marker="o", label="raw linear")
plt.plot(x_w, pred_quad_w, marker="s", label="with x²")
plt.title("1: feature map changes what linear means")
plt.xlabel("raw x"); plt.ylabel("target y"); plt.legend(); plt.show()

▶ What you'll see: the raw linear fit cuts through the curve, while the engineered representation matches every point.

*Why it's done this way:* a linear model predicts `wᵀz`, not `wᵀx`. Choosing `z = [1, x, x²]` lets the coefficient vector place separate weights on level, slope, and curvature. The model is still linear in parameters, so least squares remains simple, but the coordinate system now contains the signal-generating shape.

### 2. Normalized scales make coefficients comparable

When two features live on very different scales, a coefficient can look small simply because the feature's units are large. Standardization maps each column to `(x - mean) / std`, so one unit means one standard deviation instead of one arbitrary measurement unit.

In [ ]:
hours_w = np.array([1., 2., 3., 4., 5.])  # small scale.
income_w = np.array([30., 45., 60., 75., 90.]) * 1000  # large scale.
X_scale_w = np.c_[hours_w, income_w]
means_w = X_scale_w.mean(axis=0)
stds_w = X_scale_w.std(axis=0)
X_norm_w = (X_scale_w - means_w) / stds_w
print("means:", means_w)
print("stds:", np.round(stds_w, 3))
assert round(float(stds_w[0]), 3) == 1.414

▶ What you'll see: income has a standard deviation thousands of times larger than hours.

In [ ]:
print("normalized column means:", np.round(X_norm_w.mean(axis=0), 6))
print("normalized column stds:", np.round(X_norm_w.std(axis=0), 6))
assert np.allclose(X_norm_w.mean(axis=0), [0, 0])
assert np.allclose(X_norm_w.std(axis=0), [1, 1])

▶ What you'll see: both engineered columns now have mean 0 and standard deviation 1.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(X_scale_w[:, 0], marker="o", label="hours raw")
plt.plot(X_scale_w[:, 1], marker="s", label="income raw")
plt.title("2: raw scales hide small-unit features"); plt.legend(); plt.show()
plt.figure(figsize=(5, 3))
plt.plot(X_norm_w[:, 0], marker="o", label="hours normalized")
plt.plot(X_norm_w[:, 1], marker="s", label="income normalized")
plt.title("2: normalized scales are comparable"); plt.legend(); plt.show()

▶ What you'll see: income dominates the raw plot, but both normalized columns vary on the same visual scale.

*Why it's done this way:* many objectives depend on dot products and penalties such as `λ||w||²`. If one feature is measured in thousands and another in ones, the same predictive change can require wildly different coefficient sizes. Normalizing makes the geometry of distances, gradients, and regularization reflect signal rather than units.

### 3. Interaction features let one variable change another's effect

A raw linear model adds separate effects: `w₁x₁ + w₂x₂`. It cannot express that the effect of one feature depends on another. An interaction column `x₁x₂` gives the model a coordinate for “both are present together.”

In [ ]:
x1_w = np.array([0., 0., 1., 1.])  # first binary condition.
x2_w = np.array([0., 1., 0., 1.])  # second binary condition.
y_int_w = 1.0 + 2.0 * x1_w + 3.0 * x2_w + 4.0 * x1_w * x2_w  # includes synergy.
Z_add_w = np.c_[np.ones(4), x1_w, x2_w]
Z_inter_w = np.c_[np.ones(4), x1_w, x2_w, x1_w * x2_w]
print("targets:", y_int_w)
print("interaction column:", x1_w * x2_w)

▶ What you'll see: only the row where both conditions equal 1 receives the interaction signal.

In [ ]:
coef_add_w = np.linalg.lstsq(Z_add_w, y_int_w, rcond=None)[0]
coef_inter_w = np.linalg.lstsq(Z_inter_w, y_int_w, rcond=None)[0]
pred_add_w = Z_add_w @ coef_add_w
pred_inter_w = Z_inter_w @ coef_inter_w
print("additive predictions:", np.round(pred_add_w, 2))
print("interaction predictions:", np.round(pred_inter_w, 2))
assert np.allclose(coef_inter_w, [1, 2, 3, 4])

▶ What you'll see: the engineered interaction model recovers the exact intercept, main effects, and synergy.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(np.arange(4) - 0.15, y_int_w, width=0.3, label="truth")
plt.bar(np.arange(4) + 0.15, pred_add_w, width=0.3, label="additive only")
plt.xticks(range(4), ["00", "01", "10", "11"])
plt.title("3: additive model misses synergy"); plt.ylabel("y"); plt.legend(); plt.show()

▶ What you'll see: the additive model cannot hit all four cells because it lacks a separate coordinate for the joint case.

*Why it's done this way:* multiplication creates a basis function that is zero unless both ingredients are active. In linear algebra terms, the design matrix gains a column that is not a simple copy of either main effect, so least squares can assign a distinct coefficient to the combined condition.

### 4. Bins convert thresholds into visible step features

Some relationships are not smooth. A rule may change after a threshold: under 3, between 3 and 7, or above 7. Binning turns a continuous value into indicator columns so a linear model can represent different levels in different regions.

In [ ]:
age_w = np.array([1., 2., 4., 6., 8., 9.])
y_step_w = np.array([1., 1., 3., 3., 6., 6.])
bin_edges_w = np.array([0., 3., 7., 10.])
bin_id_w = np.digitize(age_w, bin_edges_w[1:-1])  # 0, 1, or 2.
B_w = np.eye(3)[bin_id_w]  # one-hot bin features.
print("bin ids:", bin_id_w)
print("bin feature matrix:\n", B_w.astype(int))

▶ What you'll see: each example activates exactly one threshold region.

In [ ]:
coef_bins_w = np.linalg.lstsq(B_w, y_step_w, rcond=None)[0]
pred_bins_w = B_w @ coef_bins_w
print("bin levels:", coef_bins_w)
print("predictions:", pred_bins_w)
assert np.allclose(coef_bins_w, [1, 3, 6])

▶ What you'll see: each bin coefficient becomes the average target value for that region.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.scatter(age_w, y_step_w, color="black", label="data")
plt.step(age_w, pred_bins_w, where="mid", color="darkorange", label="binned fit")
plt.title("4: bins represent threshold jumps")
plt.xlabel("raw value"); plt.ylabel("target"); plt.legend(); plt.show()

▶ What you'll see: the binned prediction follows the step pattern instead of forcing one straight line.

*Why it's done this way:* one-hot bins replace a single continuous coordinate with local basis functions. The fitted coefficient for a bin is shared by examples inside that interval, which captures threshold behavior while still pooling data within each region.

### 5. Counts and validation cost keep engineered flexibility honest

More features can lower training loss by giving the model extra degrees of freedom. Feature engineering therefore needs a decision score, not just a raw fit: empirical risk plus a cost or regularization term, checked against alternatives.

In [ ]:
losses_w = np.array([0.180, 0.148, 0.454])  # verified toy per-example losses from the lesson.
R_s_w = float(losses_w.mean())
cost_w = 0.080
score_w = R_s_w + cost_w
print("empirical risk:", round(R_s_w, 3))
print("score with cost:", round(score_w, 3))
assert round(R_s_w, 3) == 0.261
assert round(score_w, 3) == 0.341

▶ What you'll see: the raw average loss is 0.261, but the selection score is 0.341 after adding feature cost.

In [ ]:
alternative_w = 0.377
gap_w = alternative_w - score_w
relative_gap_w = gap_w / alternative_w
stable_w = 0.80 * score_w
final_scores_w = np.array([score_w, alternative_w, stable_w])
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))
print("stabilized score:", round(stable_w, 3))
print("best score:", round(float(final_scores_w.min()), 3))
assert round(gap_w, 3) == 0.036
assert round(stable_w, 3) == 0.273

▶ What you'll see: the stabilized representation wins after the full score, not the raw training number, is compared.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["engineered", "flexible alt", "stabilized"], final_scores_w, color=["steelblue", "gray", "seagreen"])
plt.ylabel("decision score (lower is better)")
plt.title("5: compare fit plus feature cost"); plt.xticks(rotation=10); plt.show()

▶ What you'll see: the stabilized score is lowest, showing why feature choices are judged by the full selection criterion.

*Why it's done this way:* empirical risk is an average over observed losses, but feature engineering changes model capacity and operational burden. Adding a cost term makes the comparison live on the same scale as the selection goal; otherwise a flexible feature map can win by memorizing convenience rather than future structure.

## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for arrays, feature matrices, least squares, and numerical checks.
import matplotlib.pyplot as plt  # Import Matplotlib for compact diagnostics of engineered features.
np.random.seed(0)  # Fix the global random seed so repeated notebook runs are reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Build a feature map

**Goal.** Turn one raw vector into a design matrix, because models learn from columns of `z = φ(x)`, not from ideas in prose. We build it in 2 steps.

In [ ]:
x_b1 = np.array([-2., -1., 0., 1., 2.])  # Create a tiny raw input vector.
phi_b1 = np.c_[np.ones_like(x_b1), x_b1, x_b1**2]  # Engineer intercept, raw value, and squared value.
print("phi shape:", phi_b1.shape)  # Inspect the number of examples and engineered columns.
print(phi_b1)  # Inspect every engineered coordinate.
assert phi_b1.shape == (5, 3)  # Verify that five examples became three-column feature rows.

▶ What you'll see: each row contains `[1, x, x²]`, the simplest nonlinear feature map.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a compact design-matrix heatmap.
plt.imshow(phi_b1, cmap="viridis", aspect="auto")  # Visualize engineered feature values as color.
plt.colorbar(label="feature value")  # Add a colorbar for numeric interpretation.
plt.title("Basic 1: feature matrix φ(x)")  # Title the diagnostic plot.
plt.xlabel("feature column")  # Label engineered columns.
plt.ylabel("example")  # Label examples.
plt.show()  # Display the matrix.

▶ What you'll see: the squared column is symmetric, unlike the raw `x` column.

👀 Takeaway: feature engineering starts by making the representation explicit as a matrix.

### Basic 2 — Fit raw versus squared features

**Goal.** Compare a raw linear feature to an added square feature, because the coordinate system determines what a linear model can express. We build it in 2 steps.

In [ ]:
x_b2 = np.array([-2., -1., 0., 1., 2.])  # Define raw scalar inputs.
y_b2 = 1 + 2 * x_b2 + 0.5 * x_b2**2  # Define a curved target.
X_raw_b2 = np.c_[np.ones_like(x_b2), x_b2]  # Build intercept plus raw x.
X_sq_b2 = np.c_[np.ones_like(x_b2), x_b2, x_b2**2]  # Add the engineered square column.
print("target:", y_b2)  # Inspect the target values before fitting.

▶ What you'll see: the target is not a straight line in raw `x`.

In [ ]:
pred_raw_b2 = X_raw_b2 @ np.linalg.lstsq(X_raw_b2, y_b2, rcond=None)[0]  # Fit with raw x only.
pred_sq_b2 = X_sq_b2 @ np.linalg.lstsq(X_sq_b2, y_b2, rcond=None)[0]  # Fit with x and x squared.
mse_raw_b2 = float(np.mean((y_b2 - pred_raw_b2) ** 2))  # Compute raw-feature MSE.
mse_sq_b2 = float(np.mean((y_b2 - pred_sq_b2) ** 2))  # Compute engineered-feature MSE.
print("MSE raw:", round(mse_raw_b2, 3), "MSE square:", round(mse_sq_b2, 3))  # Compare fit quality.
assert round(mse_raw_b2, 3) == 0.7  # Verify the raw model leaves curvature error.
assert round(mse_sq_b2, 10) == 0.0  # Verify the square feature matches the generated target.

In [ ]:
coef_raw_b2 = np.linalg.lstsq(X_raw_b2, y_b2, rcond=None)[0]  # Reuse the raw design to recover line coefficients.
coef_sq_b2 = np.linalg.lstsq(X_sq_b2, y_b2, rcond=None)[0]  # Reuse the squared design to recover curve coefficients.
x_grid_b2 = np.linspace(x_b2.min(), x_b2.max(), 100)  # Create a smooth plotting grid over the same inputs.
y_raw_grid_b2 = np.c_[np.ones_like(x_grid_b2), x_grid_b2] @ coef_raw_b2  # Evaluate the raw-feature line.
y_sq_grid_b2 = np.c_[np.ones_like(x_grid_b2), x_grid_b2, x_grid_b2**2] @ coef_sq_b2  # Evaluate the squared-feature curve.
plt.figure(figsize=(4, 3))  # Compare how each feature set fits the same target points.
plt.scatter(x_b2, y_b2, color="black", label="target")  # Plot the observed training targets.
plt.plot(x_grid_b2, y_raw_grid_b2, label="raw x fit")  # Draw the best straight-line fit.
plt.plot(x_grid_b2, y_sq_grid_b2, label="x and x² fit")  # Draw the engineered quadratic fit.
plt.xlabel("x")  # Label the input axis.
plt.ylabel("y")  # Label the target axis.
plt.title("Basic 2: raw line vs squared curve")  # Name the comparison.
plt.legend()  # Identify the two fitted models.
plt.show()

▶ What you'll see: target dots sit on the squared-feature curve, while the raw-feature line misses the bend.

▶ What you'll see: the square feature removes the error because it matches the target's hidden formula.

👀 Takeaway: adding the right feature can make a simple model fit a pattern it could not see before.

### Basic 3 — Center one feature

**Goal.** Subtract a feature mean, because centered features separate baseline level from variation around that baseline. We build it in 2 steps.

In [ ]:
x_b3 = np.array([2., 4., 6., 8.])  # Create one measured feature.
mean_b3 = float(np.mean(x_b3))  # Compute the sample mean.
centered_b3 = x_b3 - mean_b3  # Move the feature to have mean zero.
print("mean:", mean_b3)  # Inspect the value being removed.
print("centered:", centered_b3)  # Inspect deviations from the mean.
assert mean_b3 == 5.0  # Verify the concrete mean.

▶ What you'll see: values become negative below the mean and positive above the mean.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a before-after centering chart.
plt.bar(["x0", "x1", "x2", "x3"], x_b3, alpha=0.6, label="raw")  # Plot raw values.
plt.plot(centered_b3, marker="o", color="crimson", label="centered")  # Overlay centered deviations.
plt.axhline(0, color="black", linewidth=0.8)  # Mark zero after centering.
plt.title("Basic 3: centering creates deviations")  # Title the plot.
plt.legend(); plt.show()  # Display the chart.

▶ What you'll see: the centered series crosses zero while preserving relative ordering.

👀 Takeaway: centering makes coefficients describe changes around an average case.

### Basic 4 — Standardize a feature

**Goal.** Divide by standard deviation, because one standardized unit means one typical amount of variation. We build it in 2 steps.

In [ ]:
x_b4 = np.array([2., 4., 6., 8.])  # Create one measured feature.
mean_b4 = x_b4.mean()  # Compute its mean.
std_b4 = x_b4.std()  # Compute its population standard deviation for this toy array.
z_b4 = (x_b4 - mean_b4) / std_b4  # Standardize to mean 0 and standard deviation 1.
print("std:", round(float(std_b4), 3))  # Inspect the scale divisor.
print("z:", np.round(z_b4, 3))  # Inspect standardized coordinates.
assert round(float(std_b4), 3) == 2.236  # Verify the concrete scale.

▶ What you'll see: the raw numbers become dimensionless z-scores.

In [ ]:
print("z mean:", round(float(z_b4.mean()), 6), "z std:", round(float(z_b4.std()), 6))  # Verify standardization.
plt.figure(figsize=(4, 3))  # Create a z-score plot.
plt.bar(range(len(z_b4)), z_b4, color="teal")  # Plot standardized values.
plt.axhline(0, color="black", linewidth=0.8)  # Mark the mean.
plt.title("Basic 4: standardized feature")  # Title the chart.
plt.show()  # Display the plot.

▶ What you'll see: standardized values are balanced around zero with unit spread.

👀 Takeaway: standardization makes features comparable across measurement units.

### Basic 5 — Create an interaction column

**Goal.** Multiply two features, because a linear model needs a separate coordinate for joint effects. We build it in 2 steps.

In [ ]:
x1_b5 = np.array([0., 0., 1., 1.])  # First condition.
x2_b5 = np.array([0., 1., 0., 1.])  # Second condition.
interaction_b5 = x1_b5 * x2_b5  # Engineer the joint feature.
X_b5 = np.c_[np.ones(4), x1_b5, x2_b5, interaction_b5]  # Build the design matrix.
print("interaction:", interaction_b5)  # Inspect where the joint condition activates.
assert interaction_b5.sum() == 1.0  # Verify only one example has both conditions active.

▶ What you'll see: only the `[1, 1]` case turns the interaction on.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a small matrix view.
plt.imshow(X_b5, cmap="viridis", aspect="auto")  # Visualize main and interaction columns.
plt.colorbar(label="feature value")  # Add a color scale.
plt.title("Basic 5: main effects plus x1×x2")  # Title the design matrix.
plt.xlabel("feature column"); plt.ylabel("example"); plt.show()  # Display the plot.

▶ What you'll see: the interaction column is sparse and distinct from either main-effect column.

👀 Takeaway: interaction features encode “the combination matters,” not just the parts separately.

### Basic 6 — Build one-hot bins

**Goal.** Convert a continuous input into bin indicators, because thresholds are easier to learn from local indicator columns. We build it in 2 steps.

In [ ]:
value_b6 = np.array([1., 2., 4., 6., 8., 9.])  # Raw continuous values.
edges_b6 = np.array([0., 3., 7., 10.])  # Define three intervals.
bin_id_b6 = np.digitize(value_b6, edges_b6[1:-1])  # Assign each value to bin 0, 1, or 2.
onehot_b6 = np.eye(3)[bin_id_b6]  # Convert bin ids to one-hot features.
print("bin ids:", bin_id_b6)  # Inspect the bin assignment.
print(onehot_b6.astype(int))  # Inspect the one-hot matrix.
assert np.all(onehot_b6.sum(axis=1) == 1)  # Verify one active bin per example.

▶ What you'll see: each raw value is replaced by exactly one active threshold-region feature.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a bin-assignment visualization.
plt.scatter(value_b6, bin_id_b6, c=bin_id_b6, cmap="viridis", s=80)  # Plot bin id by raw value.
plt.yticks([0, 1, 2])  # Show discrete bin labels.
plt.title("Basic 6: continuous values into bins")  # Title the plot.
plt.xlabel("raw value"); plt.ylabel("bin id"); plt.show()  # Display the scatter.

▶ What you'll see: values below 3, between 3 and 7, and above 7 land in different bins.

👀 Takeaway: binning trades smoothness for flexible threshold-specific levels.

### Basic 7 — Count category occurrences

**Goal.** Turn repeated categorical events into counts, because many examples are sets or histories rather than single numbers. We build it in 2 steps.

In [ ]:
events_b7 = np.array([0, 2, 2, 1, 2, 0])  # A tiny event history with category ids 0, 1, and 2.
counts_b7 = np.bincount(events_b7, minlength=3)  # Count how often each category appears.
print("counts:", counts_b7)  # Inspect count features.
assert np.array_equal(counts_b7, np.array([2, 1, 3]))  # Verify exact category counts.

▶ What you'll see: category 2 appears most often, category 1 least often.

In [ ]:
freq_b7 = counts_b7 / counts_b7.sum()  # Convert counts to frequencies for scale-free comparison.
print("frequencies:", np.round(freq_b7, 3))  # Inspect normalized count features.
plt.figure(figsize=(4, 3))  # Create a count chart.
plt.bar(["cat0", "cat1", "cat2"], counts_b7, color="purple")  # Plot category counts.
plt.title("Basic 7: count features")  # Title the chart.
plt.ylabel("count"); plt.show()  # Display the bar chart.

▶ What you'll see: the count vector summarizes a variable-length event list as fixed-length features.

👀 Takeaway: count features convert histories into numeric columns a model can consume.

### Basic 8 — Average empirical losses

**Goal.** Compute empirical risk from per-example losses, because feature choices are judged by average loss over examples. We build it in 2 steps.

In [ ]:
losses_b8 = np.array([0.180, 0.148, 0.454])  # Use the verified toy losses from the lesson.
risk_b8 = float(np.mean(losses_b8))  # Average them into empirical risk.
print("sum losses:", round(float(losses_b8.sum()), 3))  # Inspect the numerator.
print("empirical risk:", round(risk_b8, 3))  # Inspect the average loss.
assert round(risk_b8, 3) == 0.261  # Verify the lesson risk number.

▶ What you'll see: three losses sum to 0.782 and average to 0.261.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a loss breakdown chart.
plt.bar(["ex0", "ex1", "ex2"], losses_b8, color="steelblue")  # Plot individual losses.
plt.axhline(risk_b8, color="crimson", linestyle="--", label="mean risk")  # Mark the empirical risk.
plt.title("Basic 8: empirical risk is an average")  # Title the plot.
plt.legend(); plt.show()  # Display the chart.

▶ What you'll see: the mean line summarizes all per-example losses into one training score.

👀 Takeaway: empirical risk is the average loss that the learning rule tries to reduce.

### Basic 9 — Add feature cost

**Goal.** Add a complexity or operational cost to raw risk, because the lowest training loss is not always the best choice. We build it in 2 steps.

In [ ]:
risk_b9 = 0.261  # Start from the verified empirical risk.
cost_b9 = 0.080  # Define a feature-engineering cost term.
score_b9 = risk_b9 + cost_b9  # Compute the decision score.
print("score:", round(score_b9, 3))  # Inspect risk plus cost.
assert round(score_b9, 3) == 0.341  # Verify the lesson score number.

▶ What you'll see: cost raises the selection score from raw fit to a more honest criterion.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a stacked-style breakdown.
plt.bar(["risk", "cost", "score"], [risk_b9, cost_b9, score_b9], color=["steelblue", "orange", "seagreen"])  # Compare components.
plt.title("Basic 9: raw fit plus cost")  # Title the plot.
plt.ylabel("score units"); plt.show()  # Display the bar chart.

▶ What you'll see: the final score is the quantity used for model selection, not the raw risk alone.

👀 Takeaway: cost terms keep extra engineered flexibility from looking free.

### Basic 10 — Compare a stabilized feature set

**Goal.** Compare baseline, flexible, and stabilized scores, because model selection uses the lowest full decision score. We build it in 2 steps.

In [ ]:
baseline_b10 = 0.341  # Decision score after cost.
flexible_b10 = 0.377  # A more flexible alternative's score.
stable_b10 = 0.80 * baseline_b10  # A stabilizing knob reduces the score by 20%.
scores_b10 = np.array([baseline_b10, flexible_b10, stable_b10])  # Collect the choices.
print("scores:", np.round(scores_b10, 3))  # Inspect all candidates.
assert round(stable_b10, 3) == 0.273  # Verify the stabilized score.

▶ What you'll see: the stabilized version has the smallest full score.

In [ ]:
winner_b10 = int(np.argmin(scores_b10))  # Select the lowest score.
print("winner index:", winner_b10)  # Inspect the selected candidate.
plt.figure(figsize=(4, 3))  # Create a comparison chart.
plt.bar(["baseline", "flexible", "stable"], scores_b10, color=["gray", "orange", "seagreen"])  # Plot candidate scores.
plt.title("Basic 10: choose by full score")  # Title the plot.
plt.ylabel("lower is better"); plt.xticks(rotation=10); plt.show()  # Display the chart.

▶ What you'll see: selection follows the smallest cost-aware score, not the most flexible representation.

👀 Takeaway: engineered features should be chosen by validation-aware decision scores.

## 🟡 Easy

### Easy 1 — Normalize a two-feature table

**Goal.** Standardize a small two-column feature table, because scale differences distort distances, gradients, and regularization. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[1., 30000.], [2., 45000.], [3., 60000.], [4., 75000.], [5., 90000.]])  # Two raw features on very different scales.
mean_e1 = X_e1.mean(axis=0)  # Compute one mean per column.
std_e1 = X_e1.std(axis=0)  # Compute one standard deviation per column.
print("means:", mean_e1)  # Inspect column centers.
print("stds:", np.round(std_e1, 3))  # Inspect column scales.

▶ What you'll see: the second feature's scale is thousands of times larger than the first.

In [ ]:
Z_e1 = (X_e1 - mean_e1) / std_e1  # Standardize both columns.
print("Z mean:", np.round(Z_e1.mean(axis=0), 6))  # Verify zero column means.
print("Z std:", np.round(Z_e1.std(axis=0), 6))  # Verify unit column standard deviations.
assert np.allclose(Z_e1.mean(axis=0), [0, 0])  # Check centering.
assert np.allclose(Z_e1.std(axis=0), [1, 1])  # Check scaling.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a standardized-feature plot.
plt.plot(Z_e1[:, 0], marker="o", label="feature 0 z")  # Plot first standardized feature.
plt.plot(Z_e1[:, 1], marker="s", label="feature 1 z")  # Plot second standardized feature.
plt.title("Easy 1: standardized columns")  # Title the plot.
plt.legend(); plt.show()  # Display the comparable-scale lines.

▶ What you'll see: after standardization, both columns occupy the same vertical range.

👀 Takeaway: normalization lets the model compare changes in standard-deviation units rather than raw units.

### Easy 2 — Fit an interaction model

**Goal.** Recover a known synergy coefficient, because interactions let a linear model express “more than the sum of parts.” We build it in 3 steps.

In [ ]:
x1_e2 = np.array([0., 0., 1., 1.])  # First binary feature.
x2_e2 = np.array([0., 1., 0., 1.])  # Second binary feature.
y_e2 = 1 + 2 * x1_e2 + 3 * x2_e2 + 4 * x1_e2 * x2_e2  # Target with true interaction coefficient 4.
X_e2 = np.c_[np.ones(4), x1_e2, x2_e2, x1_e2 * x2_e2]  # Include intercept, main effects, and interaction.
print("design matrix:\n", X_e2.astype(int))  # Inspect engineered columns.

▶ What you'll see: the last column activates only for the joint case.

In [ ]:
coef_e2 = np.linalg.lstsq(X_e2, y_e2, rcond=None)[0]  # Fit the linear model in engineered coordinates.
pred_e2 = X_e2 @ coef_e2  # Predict on the training examples.
print("coefficients:", np.round(coef_e2, 3))  # Inspect recovered effects.
assert np.allclose(coef_e2, [1, 2, 3, 4])  # Verify exact recovery of the data-generating coefficients.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a coefficient plot.
plt.bar(["bias", "x1", "x2", "x1×x2"], coef_e2, color="teal")  # Plot learned coefficients.
plt.title("Easy 2: learned interaction coefficient")  # Title the plot.
plt.ylabel("weight"); plt.show()  # Display the coefficients.

▶ What you'll see: the interaction bar has weight 4, the extra joint effect.

👀 Takeaway: an interaction feature gives the model one coefficient for the combined condition.

### Easy 3 — Fit binned step levels

**Goal.** Learn separate levels for threshold regions, because a single slope is a poor representation for stepwise behavior. We build it in 3 steps.

In [ ]:
x_e3 = np.array([1., 2., 4., 6., 8., 9.])  # Raw continuous values.
y_e3 = np.array([1., 1., 3., 3., 6., 6.])  # Stepwise target values.
bins_e3 = np.digitize(x_e3, [3., 7.])  # Assign low, middle, high bins.
B_e3 = np.eye(3)[bins_e3]  # One-hot encode bins.
print("bins:", bins_e3)  # Inspect threshold assignments.

▶ What you'll see: the six examples split into three two-example regions.

In [ ]:
levels_e3 = np.linalg.lstsq(B_e3, y_e3, rcond=None)[0]  # Fit one coefficient per bin.
pred_e3 = B_e3 @ levels_e3  # Predict by selecting the active bin level.
print("levels:", levels_e3)  # Inspect learned bin averages.
assert np.allclose(levels_e3, [1, 3, 6])  # Verify exact step levels.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a step fit plot.
plt.scatter(x_e3, y_e3, color="black", label="truth")  # Plot observed step targets.
plt.step(x_e3, pred_e3, where="mid", color="orange", label="binned prediction")  # Plot binned predictions.
plt.title("Easy 3: binned feature fit")  # Title the figure.
plt.legend(); plt.show()  # Display the comparison.

▶ What you'll see: the binned model exactly follows the three flat regions.

👀 Takeaway: bins are useful when the target changes by region instead of by one global slope.

### Easy 4 — Build count and log-count features

**Goal.** Compare raw counts with log-counts, because repeated events often have diminishing returns. We build it in 3 steps.

In [ ]:
histories_e4 = [np.array([0, 0, 2, 2, 2]), np.array([1, 2]), np.array([0, 1, 1, 1, 2, 2])]  # Three variable-length histories.
counts_e4 = np.vstack([np.bincount(h_e4, minlength=3) for h_e4 in histories_e4])  # Convert each history to fixed-length counts.
print("counts:\n", counts_e4)  # Inspect count features.
assert counts_e4.shape == (3, 3)  # Verify three examples by three categories.

▶ What you'll see: variable-length histories become a rectangular feature matrix.

In [ ]:
log_counts_e4 = np.log1p(counts_e4)  # Apply log(1+count) to reduce dominance of repeated events.
print("log-counts:\n", np.round(log_counts_e4, 3))  # Inspect transformed counts.
assert round(float(log_counts_e4[0, 2]), 3) == 1.386  # Verify log(1+3).

In [ ]:
plt.figure(figsize=(5, 3))  # Create a grouped count comparison.
plt.plot(counts_e4[0], marker="o", label="raw counts ex0")  # Plot raw counts for one example.
plt.plot(log_counts_e4[0], marker="s", label="log counts ex0")  # Plot log-counts for the same example.
plt.title("Easy 4: log-count compression")  # Title the chart.
plt.xticks([0, 1, 2], ["cat0", "cat1", "cat2"]); plt.legend(); plt.show()  # Display category features.

▶ What you'll see: log-counting keeps ordering but compresses the largest repeated category.

👀 Takeaway: count features summarize histories, and log transforms encode diminishing marginal evidence.

### Easy 5 — Score feature maps with cost

**Goal.** Compare three feature maps using risk plus cost, because a more flexible map must justify its added complexity. We build it in 3 steps.

In [ ]:
risks_e5 = np.array([0.261, 0.297, 0.241])  # Raw validation-like risks for three feature maps.
costs_e5 = np.array([0.080, 0.010, 0.160])  # Complexity or operational costs for those maps.
names_e5 = np.array(["engineered", "simple", "wide"])
scores_e5 = risks_e5 + costs_e5  # Compute full decision scores.
print("scores:", np.round(scores_e5, 3))  # Inspect risk plus cost.
assert round(float(scores_e5[0]), 3) == 0.341  # Verify the lesson score for the engineered map.

▶ What you'll see: the lowest raw risk is not automatically the lowest cost-aware score.

In [ ]:
best_e5 = int(np.argmin(scores_e5))  # Select by full score.
print("best feature map:", names_e5[best_e5])  # Inspect the selected representation.
print("best score:", round(float(scores_e5[best_e5]), 3))  # Inspect the winning score.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a model-selection plot.
plt.bar(names_e5, scores_e5, color=["steelblue", "gray", "orange"])  # Plot full scores.
plt.title("Easy 5: feature-map selection score")  # Title the plot.
plt.ylabel("risk + cost"); plt.show()  # Display the comparison.

▶ What you'll see: the selected map balances fit and cost rather than maximizing flexibility.

👀 Takeaway: feature engineering is a model-selection decision, not just a training-loss contest.

## 🔴 Advanced

### Advanced 1 — Degree sweep with validation

**Goal.** Compare polynomial degrees on train and validation data, because higher-degree features can overfit small samples. We build it in 4 steps.

In [ ]:
x_train_a1 = np.array([-2., -1., 0., 1., 2.])  # Training inputs.
y_train_a1 = 1 + 2 * x_train_a1 + 0.5 * x_train_a1**2  # Training targets from a quadratic rule.
x_val_a1 = np.array([-1.5, -0.5, 0.5, 1.5])  # Validation inputs between training points.
y_val_a1 = 1 + 2 * x_val_a1 + 0.5 * x_val_a1**2  # Validation targets from the same rule.
degrees_a1 = np.array([1, 2, 4])  # Test underfit, correct, and flexible degrees.
print("degrees:", degrees_a1)  # Inspect the sweep.

▶ What you'll see: the sweep compares three feature-map capacities.

In [ ]:
train_mse_a1 = []  # Store training errors.
val_mse_a1 = []  # Store validation errors.
for deg_a1 in degrees_a1:  # Loop over polynomial degrees.
    Xtr_a1 = np.vstack([x_train_a1**p_a1 for p_a1 in range(deg_a1 + 1)]).T  # Build polynomial train features.
    Xva_a1 = np.vstack([x_val_a1**p_a1 for p_a1 in range(deg_a1 + 1)]).T  # Build polynomial validation features.
    coef_a1 = np.linalg.lstsq(Xtr_a1, y_train_a1, rcond=None)[0]  # Fit least squares.
    train_mse_a1.append(float(np.mean((y_train_a1 - Xtr_a1 @ coef_a1) ** 2)))  # Record train MSE.
    val_mse_a1.append(float(np.mean((y_val_a1 - Xva_a1 @ coef_a1) ** 2)))  # Record validation MSE.
print("train MSE:", np.round(train_mse_a1, 4))  # Inspect training errors.
print("val MSE:", np.round(val_mse_a1, 4))  # Inspect validation errors.
assert round(train_mse_a1[0], 3) == 0.7  # Verify degree-1 underfit error.

In [ ]:
best_degree_a1 = int(degrees_a1[int(np.argmin(val_mse_a1))])  # Choose degree by validation error.
print("best degree:", best_degree_a1)  # Inspect selected feature complexity.
assert best_degree_a1 == 2  # Verify the quadratic map is selected.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a degree-sweep plot.
plt.plot(degrees_a1, train_mse_a1, marker="o", label="train")  # Plot train error.
plt.plot(degrees_a1, val_mse_a1, marker="s", label="validation")  # Plot validation error.
plt.title("Advanced 1: polynomial feature validation")  # Title the plot.
plt.xlabel("degree"); plt.ylabel("MSE"); plt.legend(); plt.show()  # Display the sweep.

▶ What you'll see: degree 1 underfits, degree 2 matches the true structure, and validation chooses the right capacity.

👀 Takeaway: validation prevents feature expansion from being rewarded merely for fitting the training table.

### Advanced 2 — RBF features from scratch

**Goal.** Build smooth localized basis features, because nonlinear patterns can be represented by distances to anchor points. We build it in 4 steps.

In [ ]:
x_a2 = np.linspace(-3, 3, 25)  # Create one-dimensional inputs.
centers_a2 = np.array([-2., 0., 2.])  # Choose three RBF anchor points.
gamma_a2 = 0.7  # Choose width parameter for Gaussian bumps.
Phi_a2 = np.exp(-gamma_a2 * (x_a2[:, None] - centers_a2[None, :]) ** 2)  # Compute RBF features.
print("Phi shape:", Phi_a2.shape)  # Inspect examples by basis functions.
assert Phi_a2.shape == (25, 3)  # Verify the feature matrix shape.

▶ What you'll see: each raw scalar becomes three smooth similarity-to-center features.

In [ ]:
y_a2 = np.sin(x_a2) + 0.2 * x_a2  # Create a smooth nonlinear target.
X_a2 = np.c_[np.ones_like(x_a2), Phi_a2]  # Add an intercept to RBF features.
coef_a2 = np.linalg.lstsq(X_a2, y_a2, rcond=None)[0]  # Fit a linear model in RBF space.
pred_a2 = X_a2 @ coef_a2  # Predict with the engineered features.
mse_a2 = float(np.mean((y_a2 - pred_a2) ** 2))  # Compute fit error.
print("RBF MSE:", round(mse_a2, 4))  # Inspect the approximation error.
assert mse_a2 < 0.05  # Verify the basis captures the broad nonlinear shape.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a basis-function plot.
for j_a2 in range(Phi_a2.shape[1]):  # Loop over basis columns.
    plt.plot(x_a2, Phi_a2[:, j_a2], label=f"center {centers_a2[j_a2]:.0f}")  # Plot each RBF feature.
plt.title("Advanced 2: Gaussian basis features")  # Title the plot.
plt.legend(); plt.show()  # Display the basis functions.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a fit plot.
plt.plot(x_a2, y_a2, "o", label="target")  # Plot target values.
plt.plot(x_a2, pred_a2, label="linear on RBF features")  # Plot engineered-feature predictions.
plt.title("Advanced 2: nonlinear fit from linear weights")  # Title the plot.
plt.legend(); plt.show()  # Display the fit.

▶ What you'll see: localized bumps combine into a smooth curve that tracks the nonlinear target.

👀 Takeaway: nonlinear feature maps let linear weights combine reusable basis functions.

### Advanced 3 — Crossed categorical counts

**Goal.** Create crossed category features, because the meaning of one category can depend on another. We build it in 4 steps.

In [ ]:
user_type_a3 = np.array([0, 0, 1, 1, 1, 0])  # Two user segments.
device_a3 = np.array([0, 1, 0, 1, 1, 0])  # Two device types.
cross_id_a3 = user_type_a3 * 2 + device_a3  # Encode four segment-device crosses.
counts_a3 = np.bincount(cross_id_a3, minlength=4)  # Count each crossed category.
print("cross ids:", cross_id_a3)  # Inspect crossed ids.
print("cross counts:", counts_a3)  # Inspect fixed-length crossed-count features.
assert np.array_equal(counts_a3, [2, 1, 1, 2])  # Verify counts.

▶ What you'll see: separate category pairs receive separate count slots.

In [ ]:
onehot_cross_a3 = np.eye(4)[cross_id_a3]  # One-hot encode crossed categories for each event.
print("one-hot crossed shape:", onehot_cross_a3.shape)  # Inspect event by crossed-feature matrix.
assert onehot_cross_a3.shape == (6, 4)  # Verify six events and four crossed categories.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a crossed-count chart.
plt.bar(["u0-d0", "u0-d1", "u1-d0", "u1-d1"], counts_a3, color="darkorange")  # Plot crossed counts.
plt.title("Advanced 3: crossed categorical counts")  # Title the plot.
plt.ylabel("count"); plt.xticks(rotation=20); plt.show()  # Display counts.

In [ ]:
main_user_counts_a3 = np.bincount(user_type_a3, minlength=2)  # Count user segments alone.
main_device_counts_a3 = np.bincount(device_a3, minlength=2)  # Count devices alone.
print("main user counts:", main_user_counts_a3, "main device counts:", main_device_counts_a3)  # Compare main and crossed summaries.

▶ What you'll see: main-effect counts hide which user segment paired with which device.

👀 Takeaway: crossed features preserve joint category information that separate counts can wash out.

### Advanced 4 — Leakage-safe normalization

**Goal.** Normalize validation data using training statistics only, because using validation statistics leaks future information into the feature map. We build it in 4 steps.

In [ ]:
train_a4 = np.array([10., 12., 14., 16., 18.])  # Training values available during fitting.
val_a4 = np.array([20., 22.])  # Future validation values.
mean_train_a4 = train_a4.mean()  # Compute training mean only.
std_train_a4 = train_a4.std()  # Compute training standard deviation only.
print("train mean/std:", round(float(mean_train_a4), 3), round(float(std_train_a4), 3))  # Inspect training stats.
assert mean_train_a4 == 14.0  # Verify concrete training mean.

▶ What you'll see: only the training sample determines the normalization constants.

In [ ]:
val_safe_a4 = (val_a4 - mean_train_a4) / std_train_a4  # Apply training statistics to validation.
combined_a4 = np.r_[train_a4, val_a4]  # Create a leaky combined array for contrast.
val_leaky_a4 = (val_a4 - combined_a4.mean()) / combined_a4.std()  # Incorrectly use all values.
print("safe val z:", np.round(val_safe_a4, 3))  # Inspect leakage-safe validation z-scores.
print("leaky val z:", np.round(val_leaky_a4, 3))  # Inspect the distorted leaky z-scores.

In [ ]:
diff_a4 = val_safe_a4 - val_leaky_a4  # Measure the impact of leakage.
print("difference:", np.round(diff_a4, 3))  # Inspect how much the representation changed.
assert np.all(diff_a4 > 0)  # Verify leaky normalization pulled future points closer to the center.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a safe-vs-leaky plot.
plt.bar(["safe v0", "safe v1", "leaky v0", "leaky v1"], np.r_[val_safe_a4, val_leaky_a4], color=["seagreen", "seagreen", "crimson", "crimson"])  # Compare z-scores.
plt.title("Advanced 4: validation normalization leakage")  # Title the plot.
plt.ylabel("validation z-score"); plt.xticks(rotation=15); plt.show()  # Display comparison.

▶ What you'll see: using validation values in the mean and standard deviation changes the validation representation itself.

👀 Takeaway: every learned preprocessing statistic must be fit on training data and then reused unchanged.

### Advanced 5 — Feature selection with regularized score

**Goal.** Sweep feature sets and choose by validation-style loss plus a size penalty, because engineered flexibility should pay rent. We build it in 4 steps.

In [ ]:
loss_by_set_a5 = np.array([0.310, 0.261, 0.244, 0.239])  # Raw losses for increasingly large feature sets.
num_features_a5 = np.array([1, 3, 6, 10])  # Feature counts for those sets.
lam_a5 = 0.016  # Cost per feature.
penalty_a5 = lam_a5 * num_features_a5  # Compute size penalties.
print("penalties:", np.round(penalty_a5, 3))  # Inspect complexity costs.
assert round(float(penalty_a5[1]), 3) == 0.048  # Verify the three-feature penalty.

▶ What you'll see: the cost rises as more engineered columns are added.

In [ ]:
score_a5 = loss_by_set_a5 + penalty_a5  # Compute full selection scores.
best_idx_a5 = int(np.argmin(score_a5))  # Choose the lowest score.
print("scores:", np.round(score_a5, 3))  # Inspect fit plus penalty.
print("best feature count:", int(num_features_a5[best_idx_a5]))  # Inspect selected complexity.
assert int(num_features_a5[best_idx_a5]) == 3  # Verify the moderate feature set wins.

In [ ]:
relative_gap_a5 = (score_a5[-1] - score_a5[best_idx_a5]) / score_a5[-1]  # Compare best to widest model.
print("relative gap vs widest:", round(float(relative_gap_a5), 3))  # Inspect evidence against the widest map.
assert relative_gap_a5 > 0  # Verify the selected set beats the widest full score.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a score-sweep plot.
plt.plot(num_features_a5, loss_by_set_a5, marker="o", label="raw loss")  # Plot raw losses.
plt.plot(num_features_a5, score_a5, marker="s", label="loss + penalty")  # Plot full scores.
plt.axvline(num_features_a5[best_idx_a5], color="crimson", linestyle="--", label="selected")  # Mark selected size.
plt.title("Advanced 5: feature count penalty")  # Title the plot.
plt.xlabel("number of features"); plt.ylabel("score"); plt.legend(); plt.show()  # Display the sweep.

▶ What you'll see: raw loss keeps improving, but the penalized score selects the smaller durable representation.

👀 Takeaway: feature engineering succeeds when the full score improves, not when columns merely accumulate.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Feature engineering changes representation so simple models can see the signal.

The lesson keeps the learner simple and moves capacity into the representation. Polynomial, interaction, and domain-ratio features let a linear classifier express boundaries that raw coordinates hide.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import log_loss
from sklearn.metrics import recall_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
np.random.seed(7)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def safe_split(X, y, test_size=0.4):
    counts = np.bincount(np.asarray(y))
    can_stratify = counts.min() >= 2
    stratify = y if can_stratify else None
    return train_test_split(X, y, test_size=test_size, random_state=0, stratify=stratify)


def scaled_train_test(X, y, test_size=0.4):
    x_tr, x_te, y_tr, y_te = safe_split(X, y, test_size=test_size)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def logistic_model(**kwargs):
    params = dict(max_iter=3000, solver="lbfgs")
    params.update(kwargs)
    return LogisticRegression(**params)


def predict_proba_or_scores(model, X, labels):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
    else:
        scores = model.decision_function(X)
        if scores.ndim == 1:
            scores = np.column_stack([-scores, scores])
        scores = scores - scores.max(axis=1, keepdims=True)
        proba = np.exp(scores)
        proba = proba / proba.sum(axis=1, keepdims=True)
    if proba.shape[1] == len(labels):
        return proba
    aligned = np.zeros((len(X), len(labels)))
    for j, label in enumerate(model.classes_):
        idx = list(labels).index(label)
        aligned[:, idx] = proba[:, j]
    aligned = np.clip(aligned, 1e-9, 1.0)
    aligned = aligned / aligned.sum(axis=1, keepdims=True)
    return aligned


def model_log_loss(model, x_te, y_te, labels):
    proba = predict_proba_or_scores(model, x_te, labels)
    return float(log_loss(y_te, proba, labels=labels))


def print_table(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    fmt = "  ".join("{:" + str(w) + "}" for w in widths)
    print(fmt.format(*headers))
    print(fmt.format(*["-" * w for w in widths]))
    for row in rows:
        print(fmt.format(*row))


def plot_summary(names, metrics, title, ylabel):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    flat = axes.ravel()
    for idx, (name, X, y) in enumerate(clf_ladder()):
        ax = flat[idx]
        sample = X[:, :2]
        ax.scatter(sample[:, 0], sample[:, 1], c=y, cmap="viridis", s=20, alpha=0.8)
        ax.set_title(name.split("(")[0].strip())
        ax.set_xticks([])
        ax.set_yticks([])
    ax = flat[-1]
    ax.plot(range(1, len(metrics) + 1), metrics, marker="o")
    ax.set_title(title)
    ax.set_xlabel("rung")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(1, len(metrics) + 1))
    fig.tight_layout()
    plt.show()


## The concept, built once on D1
The lesson formula is $$z=\phi(x),\qquad \hat y=f(z)$$
We first reproduce the lesson arithmetic exactly, then reuse the corresponding real technique below.

In [ ]:

def feature_engineering_method(losses, cost, alternative):
    risk = float(np.mean(losses))
    score = risk + cost
    gap = alternative - score
    return risk, score, gap

losses = np.array([0.180, 0.148, 0.454])
risk, score, gap = feature_engineering_method(losses, 0.080, 0.377)
print(f"lesson risk={risk:.3f}, score={score:.3f}, gap={gap:.3f}")
assert round(risk, 3) == 0.261
assert round(score, 3) == 0.341
assert round(gap, 3) == 0.036


Now connect the arithmetic to a reusable model-selection habit: compute the validation metric, add any stated cost, and compare the gap to an alternative. The assert guards make the notebook reproducible.

In [ ]:
lesson_losses = np.array([0.18, 0.148, 0.454])
lesson_risk, lesson_score, lesson_gap = feature_engineering_method(lesson_losses, 0.080, 0.377)
print(round(lesson_risk, 3), round(lesson_score, 3), round(lesson_gap, 3))

## The dataset ladder
The same code runs from a hand-built D1 through real D4/D5 data. Each rung reports shape, classes, and a tiny sample so leakage, search, feature work, imbalance, and strategy choices stay inspectable.

In [ ]:

rows = []
for idx, (name, X, y) in enumerate(clf_ladder(), 1):
    classes, counts = np.unique(y, return_counts=True)
    class_summary = ", ".join([f"{cls}:{cnt}" for cls, cnt in zip(classes, counts)])
    rows.append([f"D{idx}", name, str(X.shape), class_summary, np.array2string(X[:2, :min(3, X.shape[1])], precision=2)])
print_table(rows, ["rung", "name", "shape", "classes", "sample"])


## Run the same method across D1–D5
One metric is collected per rung so the summary curve is comparable.

In [ ]:

def engineer_features(X):
    base = np.asarray(X, dtype=float)
    poly = PolynomialFeatures(degree=2, include_bias=False)
    expanded = poly.fit_transform(base)
    first = base[:, [0]]
    second = base[:, [1]] if base.shape[1] > 1 else first
    ratio = first / (np.abs(second) + 1.0)
    log_abs = np.log1p(np.abs(base))
    return np.column_stack([expanded, ratio, log_abs])


def feature_engineering_experiment(X, y):
    x_tr, x_te, y_tr, y_te = safe_split(X, y)
    raw = Pipeline([
        ("scale", StandardScaler()),
        ("model", logistic_model())
    ])
    raw.fit(x_tr, y_tr)
    raw_acc = raw.score(x_te, y_te)
    z_tr = engineer_features(x_tr)
    z_te = engineer_features(x_te)
    engineered = Pipeline([
        ("scale", StandardScaler()),
        ("model", logistic_model(C=0.5))
    ])
    engineered.fit(z_tr, y_tr)
    engineered_acc = engineered.score(z_te, y_te)
    return raw_acc, engineered_acc, z_tr.shape[1]

rows = []
metrics = []
feature_results = []
for idx, (name, X, y) in enumerate(clf_ladder(), 1):
    raw_acc, engineered_acc, feature_count = feature_engineering_experiment(X, y)
    metrics.append(engineered_acc)
    feature_results.append((name, raw_acc, engineered_acc, feature_count))
    rows.append([f"D{idx}", str(X.shape[1]), str(feature_count), f"{raw_acc:.3f}", f"{engineered_acc:.3f}"])
print_table(rows, ["rung", "raw_d", "engineered_d", "raw_acc", "engineered_acc"])


## Results visualization
The first five panels preview the ladder data; the last panel tracks the metric from D1 to D5.

In [ ]:

plot_summary([r[0] for r in feature_results], metrics, "engineered accuracy vs rung", "accuracy")


## Pitfall on the hardest rung
D5 is where a shortcut can look most convincing. The cell reproduces the wrong behavior and then applies the safer fix.

In [ ]:

name, X, y = clf_ladder()[-1]
x_tr, x_te, y_tr, y_te = safe_split(X, y)
z_tr = engineer_features(x_tr)
z_te = engineer_features(x_te)
weak = logistic_model(C=100.0)
weak.fit(z_tr, y_tr)
weak_acc = weak.score(z_te, y_te)
fixed = Pipeline([
    ("scale", StandardScaler()),
    ("model", logistic_model(C=0.5))
])
fixed.fit(z_tr, y_tr)
fixed_acc = fixed.score(z_te, y_te)
print(f"unscaled high-capacity engineered D5 accuracy: {weak_acc:.3f}")
print(f"scaled and regularized engineered D5 accuracy: {fixed_acc:.3f}")
assert fixed_acc >= weak_acc



## Evaluate it + Practice
- Compare the reported metric with a no-skill baseline before trusting the method.
- Run a cheap sanity check: shuffle labels or remove the key idea and confirm performance drops.
- Ablate the technique in the table above and inspect whether D5 changes more than D1.
- Watch failure signals: unstable validation numbers, suspiciously perfect scores, or a gap that grows with complexity.

Practice prompts:
1. Change one hyperparameter or preprocessing choice and rerun the ladder.


2. Add a small amount of label noise to D3 and explain which metric moved first.

3. On D5, write one sentence explaining whether the method is reducing bias, variance, or evaluation error.